# 🦀 Day 7: Mini-Project — Personal Dev-Tools CLI
---

Today we build a **Swiss-army knife CLI** combining everything from this week.

- **find**: Recursive file search with regex and extension filters
- **count**: Line/word/char counter (like `wc`) with multiple files
- **replace**: Search & replace in files with regex support
- **stats**: Project statistics — files by type, total lines, largest files
- **format**: Convert between config formats (exercise)

Uses: clap, regex, walkdir, serde, serde_json, toml

In [ ]:
:dep clap = { version = "4", features = ["derive"] }
:dep regex = "1"
:dep walkdir = "2"
:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep toml = "0.8"

In [ ]:
use clap::{Parser, Subcommand};
use regex::Regex;
use walkdir::WalkDir;
use std::fs;
use std::path::Path;

#[derive(Parser)]
#[command(name = "devtools", about = "Personal dev-tools CLI")]
struct Cli {
    #[command(subcommand)]
    command: Command,
}

#[derive(Subcommand)]
enum Command {
    Find {
        #[arg(default_value = ".")]
        path: String,
        #[arg(short, long)]
        pattern: Option<String>,
        #[arg(short, long)]
        ext: Option<String>,
    },
    Count {
        paths: Vec<String>,
    },
    Replace {
        pattern: String,
        replacement: String,
        #[arg(default_value = ".")]
        path: String,
    },
    Stats {
        #[arg(default_value = ".")]
        path: String,
    },
}

let args = Cli::parse_from(["devtools", "stats", "."]);
println!("Command: {:?}", args.command);

In [ ]:
// find: recursive search with regex and extension filters
fn cmd_find(path: &str, pattern: Option<&str>, ext: Option<&str>) {
    let re = pattern.and_then(|p| Regex::new(p).ok());
    for entry in WalkDir::new(path).into_iter().filter_map(|e| e.ok()) {
        if entry.file_type().is_file() {
            let p = entry.path();
            let name = p.file_name().and_then(|n| n.to_str()).unwrap_or("");
            let ext_ok = ext.map_or(true, |e| p.extension().map_or(false, |x| x == e));
            let pattern_ok = re.as_ref().map_or(true, |r| r.is_match(name));
            if ext_ok && pattern_ok {
                println!("{}", p.display());
            }
        }
    }
}

cmd_find(".", Some(r"\.rs$"), Some("rs"));

In [ ]:
// count: line/word/char counter (like wc)
fn count_file(path: &Path) -> (usize, usize, usize) {
    let content = fs::read_to_string(path).unwrap_or_default();
    let lines = content.lines().count();
    let words = content.split_whitespace().count();
    let chars = content.chars().count();
    (lines, words, chars)
}

let (lines, words, chars) = count_file(Path::new("Cargo.toml"));
println!("Cargo.toml: {} lines, {} words, {} chars", lines, words, chars);

In [ ]:
// stats: files by type, total lines, largest files
use std::collections::HashMap;

fn cmd_stats(path: &str) {
    let mut by_ext: HashMap<String, (usize, u64)> = HashMap::new();
    let mut total_lines = 0usize;
    let mut files_with_size: Vec<(String, u64)> = vec![];

    for entry in WalkDir::new(path).into_iter().filter_map(|e| e.ok()) {
        if entry.file_type().is_file() {
            let p = entry.path();
            let ext = p.extension().and_then(|e| e.to_str()).unwrap_or("none").to_string();
            let size = entry.metadata().map(|m| m.len()).unwrap_or(0);
            let (count, total) = by_ext.entry(ext).or_insert((0, 0));
            *count += 1;
            *total += size;
            if let Some(s) = p.to_str() {
                files_with_size.push((s.to_string(), size));
            }
            if let Ok(c) = fs::read_to_string(p) {
                total_lines += c.lines().count();
            }
        }
    }

    println!("Total lines: {}", total_lines);
    println!("By extension:");
    for (ext, (count, _)) in &by_ext {
        println!("  .{}: {} files", ext, count);
    }
    files_with_size.sort_by(|a, b| b.1.cmp(&a.1));
    println!("Largest files (top 3):");
    for (p, s) in files_with_size.iter().take(3) {
        println!("  {} ({} bytes)", p, s);
    }
}

cmd_stats(".");

In [ ]:
// replace: find and replace with regex (conceptual — dry run)
fn cmd_replace(path: &str, pattern: &str, replacement: &str, dry_run: bool) {
    let re = Regex::new(pattern).expect("Invalid regex");
    for entry in WalkDir::new(path).into_iter().filter_map(|e| e.ok()) {
        if entry.file_type().is_file() {
            if let Ok(content) = fs::read_to_string(entry.path()) {
                let new_content = re.replace_all(&content, replacement);
                if new_content != content {
                    println!("Would update: {}", entry.path().display());
                    if !dry_run {
                        let _ = fs::write(entry.path(), new_content.as_ref());
                    }
                }
            }
        }
    }
}

println!("Replace uses Regex::replace_all on file contents");

## Cargo project structure

```
devtools/
├── Cargo.toml   # clap, regex, walkdir, serde, serde_json, toml
└── src/
    └── main.rs  # match args.command { Find => ..., Count => ..., ... }
```

Run: `cargo run -- find . -e rs` or `cargo run -- stats .`

## 📝 Exercise: Add `format` subcommand

Add a `format` subcommand that converts between config formats:
- `devtools format input.json toml` → read JSON, output TOML
- Use serde + toml + serde_json (and optionally serde_yaml, ron)

In [ ]:
// YOUR CODE HERE
// Format {
//     input: PathBuf,
//     output_format: String,  // toml, json, yaml, ron
// }
// Read file, detect format from extension, parse, serialize to output_format

## 🎯 Key Takeaways

1. **Week 17 Complete!** You built a multi-command dev-tools CLI
2. **find**: walkdir + regex + extension filter for recursive file search
3. **count**: Read files, count lines/words/chars (wc-like)
4. **stats**: Aggregate by extension, total lines, largest files
5. **replace**: Regex replace across files (use dry-run for safety)
6. clap subcommands tie it all together; each command is a focused function

---
**Next week:** Macros!